# Molecular Ground State via VQE using `divi`

> 🚀 **Don't choke your local machine.** Qoro is giving away **$100 in free cloud compute credits.**
> Get your API key at **[dash.qoroquantum.net](https://dash.qoroquantum.net)** to run this at scale.

## Why Cloud?

Computing the H₂ potential energy surface means running **2 ansätze × 12 bond lengths = 24 independent VQE runs.** Each VQE has its own optimizer loop with hundreds of circuit evaluations. Sequentially, that's 24× the wait. QoroService dispatches **all 24 VQE instances in parallel.**

## Step 0 — Install & Authenticate

```bash
pip install qoro-divi matplotlib
```

In [ ]:
import time

import numpy as np

# Set your API key (get one free at https://dash.qoroquantum.net)
# Create a .env file at the root and set the QORO_API_TOKEN

from molecular_ground_state import (
    build_h2_hamiltonians,
    get_ansatze,
    run_sweep,
    extract_pes_data,
    print_results,
    plot_pes,
)

from divi.backends import QoroService, ParallelSimulator, JobConfig

---

## Phase 1 — Local PES (5 bond lengths, 10 VQE runs)

5 bond lengths × 2 ansätze = 10 VQE runs. Fast enough locally to prove the algorithm works.

In [ ]:
BOND_LENGTHS_LOCAL = np.linspace(0.3, 2.5, 5).tolist()

local_backend = ParallelSimulator(shots=5_000)
print("💻 Using local ParallelSimulator")

print(f"\nBuilding H₂ Hamiltonians for {len(BOND_LENGTHS_LOCAL)} bond lengths...")
hamiltonians_local = build_h2_hamiltonians(BOND_LENGTHS_LOCAL)
print(f"  Done — 4 qubits per Hamiltonian")

In [ ]:
ansatze = get_ansatze()
print(f"Ansätze: {[a.name for a in ansatze]}")

t0 = time.time()
sweep_local = run_sweep(
    hamiltonians=hamiltonians_local,
    ansatze=ansatze,
    n_electrons=2,
    max_iterations=15,
    backend=local_backend,
)
local_time = time.time() - t0

print(f"\n✅ Phase 1 complete in {local_time:.1f}s")

In [ ]:
pes_data_local = extract_pes_data(sweep_local)
print_results(pes_data_local, sweep_local)
plot_pes(pes_data_local, save_path="h2_pes_local.png")

---

## Phase 2 — High-Resolution PES with QoroService (24 VQE runs)

12 bond lengths × 2 ansätze = 24 VQE runs, all dispatched to QoroService in parallel.

Create a `.env` file in the repo root:
```
QORO_API_KEY="your_api_key_here"
```
👉 **[Get your free API key →](https://dash.qoroquantum.net)**

In [ ]:
BOND_LENGTHS_CLOUD = np.linspace(0.3, 2.5, 12).tolist()
cloud_backend = QoroService(job_config=JobConfig(shots=10_000))

print(f"☁️  Dispatching {len(BOND_LENGTHS_CLOUD) * len(ansatze)} VQE instances to Qoro Maestro...")
print(f"   {len(BOND_LENGTHS_CLOUD)} bond lengths × {len(ansatze)} ansätze")

hamiltonians_cloud = build_h2_hamiltonians(BOND_LENGTHS_CLOUD)

t0 = time.time()
sweep_cloud = run_sweep(
    hamiltonians=hamiltonians_cloud,
    ansatze=ansatze,
    n_electrons=2,
    max_iterations=25,
    backend=cloud_backend,
)
cloud_time = time.time() - t0

pes_data_cloud = extract_pes_data(sweep_cloud)
print_results(pes_data_cloud, sweep_cloud)
plot_pes(pes_data_cloud, save_path="h2_pes_cloud.png")

print(f"
⚡ Local  (Phase 1): {local_time:.1f}s for {len(BOND_LENGTHS_LOCAL) * len(ansatze)} VQE runs")
print(f"⚡ Cloud  (Phase 2): {cloud_time:.1f}s for {len(BOND_LENGTHS_CLOUD) * len(ansatze)} VQE runs")

---

## Ready for Larger Molecules?

24 VQE runs in parallel — full PES in a fraction of the time. That's the power of QoroService.

👉 **[Get your free API key at dash.qoroquantum.net](https://dash.qoroquantum.net)** — $100 in credits, no credit card required.